In [1]:
## Mirrors the cellPLM-embedding-wrapper notebook: load a dataset, run the
## scGPT_human zero-shot model, and write the per-cell latents to a small h5ad
## that benchmark.ipynb can align by obs_names.

In [3]:
import sys, os
sys.path.insert(0, os.path.abspath('../../models/scGPT'))

import warnings
warnings.filterwarnings('ignore')

import random
#import hdf5plugin
import numpy as np
import torch
import anndata as ad
from pathlib import Path
import scgpt as scg

In [4]:
DATA_PATH = '../../data/cellPLM/data/gse155468.h5ad'
MODEL_DIR = Path('../../models/scGPT/save/scGPT_human')   # populated by env/setup_scgpt.sh
DEVICE    = 'cuda:0'
SEED      = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [5]:
data = ad.read_h5ad(DATA_PATH)
data.obs_names_make_unique()

# scGPT looks up gene symbols via a column in adata.var; gse155468 stores symbols
# in var_names with no var columns, so mirror them into 'gene_name'.
data.var['gene_name'] = data.var_names
print(f'data: {data.shape}, var cols: {data.var.columns.tolist()}')

data: (48082, 12382), var cols: ['gene_name']


In [6]:
# Zero-shot embedding via scGPT_human. embed_data() filters genes to those in
# the model vocab, tokenizes, runs the encoder, and writes per-cell latents to
# adata.obsm['X_scGPT']. Returns a new AnnData (return_new_adata=True) so the
# original `data` is left untouched for downstream cells.
embed_adata = scg.tasks.embed_data(
    data,
    MODEL_DIR,
    gene_col='gene_name',
    batch_size=64,
    device=DEVICE,
    return_new_adata=True,
)

scGPT - INFO - match 11276/12382 genes in vocabulary of size 60697.


Embedding cells: 100%|██████████| 752/752 [00:59<00:00, 12.72it/s]


In [7]:
embedding = embed_adata.X if embed_adata.X is not None else embed_adata.obsm['X_scGPT']
embedding = np.asarray(embedding, dtype=np.float32)
print('embedding shape:', embedding.shape, 'dtype:', embedding.dtype)
assert embedding.shape[0] == data.n_obs, (embedding.shape, data.n_obs)

embedding shape: (48082, 512) dtype: float32


In [8]:
# Match cellPLM-embedding-wrapper's output layout so benchmark.ipynb can align
# by obs_names: X = (n_cells, d) float32, obs_names = original cell IDs,
# var_names = ['dim_0', 'dim_1', ...].
embedding_adata = ad.AnnData(X=embedding)
embedding_adata.obs_names = data.obs_names
embedding_adata.var_names = [f'dim_{i}' for i in range(embedding.shape[1])]
embedding_adata.write_h5ad('gse155468_embedding.h5ad')
print('wrote gse155468_embedding.h5ad', embedding_adata.shape)

wrote gse155468_embedding.h5ad (48082, 512)
